# 02 Cleaning

Apply the Beijing PM2.5 cleaning pipeline, document every transformation, and export the processed dataset to `data/processed/`.

**Input:**  `data/raw/PRSA_data_2010.1.1-2014.12.31.csv`  
**Output:** `data/processed/beijing_pm25_cleaned.csv`

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import clean_beijing_pm25

## 1. Load Raw Data

In [ ]:
RAW_PATH       = PROJECT_ROOT / 'data/raw/PRSA_data_2010.1.1-2014.12.31.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/beijing_pm25_cleaned.csv'

df_raw = pd.read_csv(RAW_PATH)
print('Raw shape:', df_raw.shape)
df_raw.head(3)

## 2. Run Cleaning Pipeline

In [ ]:
df = clean_beijing_pm25(df_raw)
print('Cleaned shape:', df.shape)
df.head(3)

## 3. Transformations Applied

| # | Transformation | Reason |
|---|---|---|
| 1 | Dropped `No` column | Row index from source file, no analytical value |
| 2 | Created `datetime` from year/month/day/hour | Single timestamp column required for time-series analysis and Tableau |
| 3 | Renamed all columns to snake_case | Consistency; removes special characters (e.g. `pm2.5` → `pm2_5`) |
| 4 | Created `pm2_5_cleaned` via forward-fill then backward-fill | Preserves original nulls in `pm2_5` while providing a complete series for analysis |
| 5 | Created `season` from `month` | Enables seasonal analysis; PM2.5 in Beijing has strong seasonal patterns |
| 6 | Created `aqi_category` from `pm2_5_cleaned` | Maps raw concentrations to US EPA categories for dashboard readability |

## 4. Verify Cleaning

In [ ]:
print('=== Missing Values After Cleaning ===')
print(df.isnull().sum())
print()
print('Note: pm2_5 retains original nulls by design. Use pm2_5_cleaned for analysis.')

In [ ]:
print('=== Derived Column Distributions ===')
print('\nseason:')
print(df['season'].value_counts())
print('\naqi_category:')
print(df['aqi_category'].value_counts())

In [ ]:
print('=== Date Range ===')
print('From:', df['datetime'].min())
print('To:  ', df['datetime'].max())
print()
print('=== pm2_5_cleaned Stats ===')
print(df['pm2_5_cleaned'].describe())

## 5. Export Processed Dataset

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to {PROCESSED_PATH}')
print(f'Rows: {len(df)} | Columns: {len(df.columns)}')